# 02. Baseline model

In this notebook I build a baseline for classifying Chinese food delivery reviews (waimai_10k).

Plan:
1. Load the data and remove duplicates with conflicting labels (as in the EDA notebook).
2. Split the data into train and test (80/20, stratified).
3. Convert text to numbers with TF-IDF on character n-grams.
4. Train a logistic regression inside a Pipeline.
5. Evaluate it, focusing on recall and F2 of the negative class (0), and compare with the dummy baseline (~0.67 accuracy).

Let's make our data clear, as we did it on EDA.

In [23]:
import pandas as pd

df = pd.read_csv("../data/waimai_10k.csv") 
mask = df.duplicated(subset="review", keep=False)
df.loc[mask, "label"] = 1
df = df.drop_duplicates(subset="review").reset_index(drop=True)
df.shape   

(11980, 2)

For the further  work with our models, split our data into train and test samples. Test sample size would be 20% of the data.

In [24]:
from sklearn.model_selection import train_test_split
X = df["review"]
y = df['label']
X_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size =0.2,
    random_state=42,
    stratify=y
)

In [25]:
print(f"Train shape: {X_train.shape}, Test shape: {x_test.shape}")

Train shape: (9584,), Test shape: (2396,)


In [26]:
print(f'Train label distribution:\n{y_train.value_counts(normalize=True)}')
print(f'Test label distribution:\n{y_test.value_counts(normalize=True)}')

Train label distribution:
label
0    0.66611
1    0.33389
Name: proportion, dtype: float64
Test label distribution:
label
0    0.66611
1    0.33389
Name: proportion, dtype: float64


The shapes and distributions of both samples are right.

## Train/test split

The data was split into train (9,584 rows) and test (2,396 rows) with `test_size=0.2`, `random_state=42` and `stratify=y`. Class proportions are the same in both parts (~66.6% negative, ~33.4% positive), so the test set reflects the real class balance. The test set is not used until the final evaluation, so there is no leakage.

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])

## Why cross-validation

I compare models and tune hyperparameters with stratified 5-fold cross-validation on the train set only. A single validation split gives a noisy estimate that depends on which reviews fall into it, while CV averages 5 estimates and shows their spread. Stratification keeps the 67/33 class ratio in every fold. The test set stays untouched until the final evaluation, so the final score is an honest estimate.

In [28]:
# Train our model 
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, fbeta_score, recall_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f2_neg = make_scorer(fbeta_score, beta=2, pos_label=0)


In [39]:
#Explore the scores for baseline model

scores = cross_validate(
    pipe, X_train, y_train, cv=cv,
    scoring={"f2_neg": f2_neg, "recall_neg": make_scorer(recall_score, pos_label=0)},
)
print(f"Mean F2 (class 0): {scores['test_f2_neg'].mean():.4f}")
print(f"Std F2 (class 0): {scores['test_f2_neg'].std():.4f}")
print(f"Mean recall (class 0): {scores['test_recall_neg'].mean():.4f}")

Mean F2 (class 0): 0.9464
Std F2 (class 0): 0.0066
Mean recall (class 0): 0.9618


In [30]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_score

scoring = {
    "f2_neg": f2_neg,
    "recall_neg": make_scorer(recall_score, pos_label=0),
    "precision_neg": make_scorer(precision_score, pos_label=0),
    "f1_macro": "f1_macro",
}

dummy = DummyClassifier(strategy="most_frequent")
dummy_scores = cross_validate(dummy, X_train, y_train, cv=cv, scoring=scoring)
model_scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)

In [31]:
print(f'dummy scores: {pd.DataFrame(dummy_scores).mean()}')
print(f'model scores: {pd.DataFrame(model_scores).mean()}')

dummy scores: fit_time              0.001498
score_time            0.011895
test_f2_neg           0.908884
test_recall_neg       1.000000
test_precision_neg    0.666110
test_f1_macro         0.399800
dtype: float64
model scores: fit_time              0.487617
score_time            0.074426
test_f2_neg           0.946390
test_recall_neg       0.961779
test_precision_neg    0.889468
test_f1_macro         0.876495
dtype: float64


## Baseline results and choice of metric

I evaluated the baseline pipeline (TF-IDF on character 1-2 grams + logistic regression) with stratified 5-fold cross-validation on the train set only, and compared it with a dummy model that always predicts the majority class (0).

| metric | dummy | model |
|---|---|---|
| F2 (class 0) | 0.909 | 0.946 |
| recall (class 0) | 1.000 | 0.962 |
| precision (class 0) | 0.666 | 0.889 |
| macro-F1 | 0.400 | 0.876 |

**Why I changed the metric.** I first planned to use recall/F2 of the negative class, because a platform wants to catch complaints. But class 0 is the majority (~67%), so a dummy model that labels everything as negative already gets recall = 1.0 and F2 = 0.909. These metrics barely separate the model from the dummy, and recall alone even favors the dummy. Macro-F1 treats both classes equally and shows a clear gap (0.40 vs 0.88), so it is a more honest main metric. I will still report precision and recall of class 0 next to it, because they show how well the model finds complaints (~89% precision, ~96% recall).

**Conclusion.** The baseline is much better than the dummy model, and the fold-to-fold spread is small (std of F2 ≈ 0.007). Next step: compare `class_weight="balanced"`, CountVectorizer and different n-gram ranges using macro-F1 in cross-validation, still without touching the test set.

In [32]:
def run_cv(model):
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    return {
        "macro_f1": s["test_f1_macro"].mean(),
        "macro_f1_std": s["test_f1_macro"].std(),
        "recall_0": s["test_recall_neg"].mean(),
        "precision_0": s["test_precision_neg"].mean(),
    }

Let's try to use different params for our models.

In [33]:
from sklearn.base import clone
pp = clone(pipe)
pp['clf'].set_params(class_weight='balanced')
run_cv(pp)


{'macro_f1': np.float64(0.88282012800934),
 'macro_f1_std': np.float64(0.00792503493361998),
 'recall_0': np.float64(0.9197992944867354),
 'precision_0': np.float64(0.9231180757721095)}

In [34]:
from sklearn.base import clone
p = clone(pipe)
p.set_params(tfidf__ngram_range=(1, 3), clf__class_weight=None)
run_cv(p)

{'macro_f1': np.float64(0.8716418822255358),
 'macro_f1_std': np.float64(0.010079521647887064),
 'recall_0': np.float64(0.9642855389419266),
 'precision_0': np.float64(0.8834800460801334)}

In [35]:
from sklearn.feature_extraction.text import CountVectorizer
p_p = clone(pipe)
p_p.set_params(tfidf=CountVectorizer(analyzer="char", ngram_range=(1, 2)))
run_cv(p_p)


{'macro_f1': np.float64(0.8827118879252979),
 'macro_f1_std': np.float64(0.008792741102730081),
 'recall_0': np.float64(0.9346797573662803),
 'precision_0': np.float64(0.9126515263620153)}

In [36]:
p_lg = clone(pipe)
C_values = [0.1, 1, 10]
for C in C_values:
    p_lg.set_params(clf=LogisticRegression(C=C))
    scores = run_cv(p_lg)
    print(f"C={C}: {scores}\n")


C=0.1: {'macro_f1': np.float64(0.7814487602885666), 'macro_f1_std': np.float64(0.010441887281550715), 'recall_0': np.float64(0.9796363439978594), 'precision_0': np.float64(0.8072899609370264)}

C=1: {'macro_f1': np.float64(0.8764954012360935), 'macro_f1_std': np.float64(0.00989039438669319), 'recall_0': np.float64(0.9617792975797999), 'precision_0': np.float64(0.8894679174293489)}

C=10: {'macro_f1': np.float64(0.8867606417745544), 'macro_f1_std': np.float64(0.00728283601713944), 'recall_0': np.float64(0.9497174510203431), 'precision_0': np.float64(0.9070919911520046)}



Try the best params combination:

In [37]:
p_b = clone(pipe)
p_b.set_params(tfidf=CountVectorizer(analyzer="char", ngram_range=(1, 2)), clf__C=10)
run_cv(p_b)

{'macro_f1': np.float64(0.8714322157391703),
 'macro_f1_std': np.float64(0.008480193930836773),
 'recall_0': np.float64(0.9224606800323054),
 'precision_0': np.float64(0.9084973558625149)}

In [38]:


results = pd.DataFrame({
    "C=0.1":          {"macro_f1": 0.7814, "macro_f1_std": 0.0104, "recall_0": 0.9796, "precision_0": 0.8073},
    "baseline (C=1)": {"macro_f1": 0.8765, "macro_f1_std": 0.0099, "recall_0": 0.9618, "precision_0": 0.8895},
    "C=10":           {"macro_f1": 0.8868, "macro_f1_std": 0.0073, "recall_0": 0.9497, "precision_0": 0.9071},
    "balanced":       {"macro_f1": 0.8828, "macro_f1_std": 0.0079, "recall_0": 0.9198, "precision_0": 0.9231},
    "ngram (1,3)":    {"macro_f1": 0.8716, "macro_f1_std": 0.0101, "recall_0": 0.9643, "precision_0": 0.8835},
    "CountVectorizer": {"macro_f1": 0.8778, "macro_f1_std": 0.0090, "recall_0": 0.9129, "precision_0": 0.9222},
    "Count + C=10":    {"macro_f1": 0.8714, "macro_f1_std": 0.0084, "recall_0": 0.9224, "precision_0": 0.9084},
}).T.sort_values("macro_f1", ascending=False)

results

,macro_f1,macro_f1_std,recall_0,precision_0
C=10,0.8868,0.0073,0.9497,0.9071
balanced,0.8828,0.0079,0.9198,0.9231
CountVectorizer,0.8778,0.0090,0.9129,0.9222
baseline (C=1),0.8765,0.0099,0.9618,0.8895
"ngram (1,3)",0.8716,0.0101,0.9643,0.8835
Count + C=10,0.8714,0.0084,0.9224,0.9084
C=0.1,0.7814,0.0104,0.9796,0.8073


## Comparison of baseline variants (5-fold CV on train)

All variants use character n-grams and logistic regression, and were compared with the same stratified folds on the train set only (the test set is untouched). The dummy model (always predicts class 0) gets macro-F1 = 0.40.

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>macro_f1</th>
      <th>macro_f1_std</th>
      <th>recall_0</th>
      <th>precision_0</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>C=10</th>
      <td>0.8868</td>
      <td>0.0073</td>
      <td>0.9497</td>
      <td>0.9071</td>
    </tr>
    <tr>
      <th>balanced</th>
      <td>0.8828</td>
      <td>0.0079</td>
      <td>0.9198</td>
      <td>0.9231</td>
    </tr>
    <tr>
      <th>CountVectorizer</th>
      <td>0.8778</td>
      <td>0.0090</td>
      <td>0.9129</td>
      <td>0.9222</td>
    </tr>
    <tr>
      <th>baseline (C=1)</th>
      <td>0.8765</td>
      <td>0.0099</td>
      <td>0.9618</td>
      <td>0.8895</td>
    </tr>
    <tr>
      <th>ngram (1,3)</th>
      <td>0.8716</td>
      <td>0.0101</td>
      <td>0.9643</td>
      <td>0.8835</td>
    </tr>
    <tr>
      <th>Count + C=10</th>
      <td>0.8714</td>
      <td>0.0084</td>
      <td>0.9224</td>
      <td>0.9084</td>
    </tr>
    <tr>
      <th>C=0.1</th>
      <td>0.7814</td>
      <td>0.0104</td>
      <td>0.9796</td>
      <td>0.8073</td>
    </tr>
  </tbody>
</table>
</div>

**What I see:**
- The best variant is `C=10` (macro-F1 0.887), followed by `class_weight="balanced"` (0.883). The difference between them is smaller than the fold-to-fold std (~0.008), so I can't say one is clearly better.
- Weak regularization (larger C) helps: `C=0.1` is much worse (0.781), `C=1` gives 0.877, `C=10` gives 0.887. The best C is at the edge of my grid, so higher values should be checked.
- `class_weight="balanced"` trades recall of class 0 (0.920 vs 0.962) for precision (0.923 vs 0.890). The model labels fewer positive reviews as negative, but misses more complaints.
- Replacing TF-IDF with CountVectorizer (0.878 vs 0.877) and using 1-3 grams (0.872) did not help, so TF-IDF with 1-2 grams stays.
- `Count + C=10` (0.871) is not better than `C=10` with TF-IDF, so combining the changes did not add anything.
- Overall all reasonable variants lie within ~0.015 macro-F1, so tuning logistic regression gives only small gains. The limit is probably in the features (character n-grams) and noisy labels.

**Next steps:**
1. Tune hyperparameters with Optuna (C on a log scale up to ~100, class_weight, ngram_range), maximizing mean macro-F1 in CV on train.
2. Try decision tree and kNN (cosine distance) on the same TF-IDF features for comparison.
3. Choose the final model, train it on the whole train set and evaluate it once on the test set.
4. Error analysis: read reviews the model gets wrong and check whether the errors come from the model or from label noise.